# Scraping of Serie A players stats

## 1. With this JSON, I extract all the Serie A teams along with their unique codes (for now only 2025-2026).
 
https://www.fotmob.com/_next/data/0Sm3sug45d5kO4fly3s6V/it/leagues/55/overview/serie.json?lng=it&id=55&tab=overview&slug=serie&season=2025-2026

In [20]:
import os
import json
import csv
import pandas as pd
import time
import requests
from datetime import datetime
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
from bs4 import BeautifulSoup

In [ ]:
csv_file_path = r'.\Datasets\PlayerStats\serieA_teams_25-26.csv'

try:
    
    with open(r".\Datasets\PlayerStats\serieA_teams_25-26.json", "r") as f:
        data = json.load(f)

    # Estrai la lista delle squadre
    teams_list = data['pageProps']['table'][0]['data']['table']['all']
    
    # 3. Salva i dati in un file CSV
    with open(csv_file_path, 'w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        
        # intestazione del file CSV
        writer.writerow(['ID Univoco', 'Nome Squadra'])
        
        # scrive nel CSV
        for team in teams_list:
            team_id = team.get('id')
            team_name = team.get('name')
            
            writer.writerow([team_id, team_name])
            
    print(f"Estrazione completata con successo! Il file è stato salvato in: {csv_file_path}")

except FileNotFoundError:
    print(f"Errore: Il file json non è stato trovato. Assicurati che sia nella stessa cartella dello script.")
except KeyError as e:
    print(f"Errore nella struttura del JSON. Chiave non trovata: {e}")
except Exception as e:
    print(f"Si è verificato un errore imprevisto: {e}")

Estrazione completata con successo! Il file è stato salvato in: .\Datasets\PlayerStats\serieA_teams_25-26.csv


<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Giacomo\AppData\Local\Temp\ipykernel_14384\2851976192.py:5: SyntaxWarning: invalid escape sequence '\D'
  csv_file_path = '.\Datasets\PlayerStats\serieA_teams_25-26.csv'


## 2. Once we have obtained the unique code and the name of the teams it is associated with, we can paste them into this link:

https://www.fotmob.com/_next/data/0Sm3sug45d5kO4fly3s6V/it/teams/CODICE/overview/SQUADRA.json?lng=it&id=CODICE&tab=overview&slug=SQUADRA

This link downloads a json containing all the team's players along with their individual statistics

<b>NOTE</b>: there are rating and seasonRating, they might be useful but the detailed stats of the player are missing <br>
<b>NOTE</b>: it does not have seasonality, or I haven't found a json for each season

In this json there are also the unique id of the player and their name

In [ ]:
# Configurazione file
input_csv_path = r'.\Datasets\PlayerStats\serieA_teams_25-26.csv'
output_csv_path = r'.\Datasets\PlayerStats\listagiocatori_serie_A_stats.csv'

In [6]:
def clean_header_string(text):
    if not text:
        return ""
    text = text.replace('\u2026', '...')
    return text.encode('latin-1', 'ignore').decode('latin-1')

In [7]:
cookie_sporco = r"""g_state={"i_l":3,"i_ll":1780908209557,"i_b":"6r+lIoupE1oLWysted3I+3aJsj0tpW3Edh60RPc0wdk","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1781364585107}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkOTllLWE5NDMtNWFjYS05YzE3LTI2MTQxNTdhMTRlOSIsImNyZWF0ZWQiOjE3Nzk3MTA2NjE2MzQsImV4aXN0aW5nIjp0cnVlfQ==; FCCDCF=%5Bnull%2Cnull%2Cnull%2C%5B%22CQkw6oAQkw6oAEsACBITCgFoAP_gAEPgABpwLItB_C7dbWFDyLZ3absEeIhHx_hjasQwAAbAAmAFTBqQoJQGwmEYNAyCtCACGAAAKmTBIQIkCAAAAUAAIIAFIADMAAyQIBAIICBAiAEJAEBACEACCoAgUQCIgIJEFEQAm…27.0.0.1%22%2C%22ccode3%22%3A%22ITA%22%2C%22ccode3NoRegion%22%3A%22ITA%22%2C%22timezone%22%3A%22Europe%2FRome%22%7D; panoramaId_expiry=1780991493427; _hjSession_2585474=eyJpZCI6ImVmMTg1Y2ZjLTAzYjYtNDU2My1hMGYyLWUzNTIyMDU5NDhjNyIsImMiOjE3ODA5MDgyMDg3MzUsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MH0=; FCNEC=%5B%5B%22AKsRol8IDA2Ltzulzvmbyb8vyXeyLKTHpSQYhq3JM1nLP6SWhWGtBOJvZ6We_UQHwghnjYq0yI3yhs4jekexwZ6E6odp6ikn2kyma7ly6i8-SxMmz3mZOqcFCCyeiWwSg3eSvS8Pxx7OCwezEoNaqy7-0Mu_Wwd9-A%3D%3D%22%5D%5D"""
cookie_pulito = clean_header_string(cookie_sporco)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Alt-Used": "www.fotmob.com",
    "Cookie": cookie_pulito
}

In [9]:
def clean_team_name(text):
    text_clean = str(text).replace(" ", "-")
    return "".join(c for c in text_clean if c.isalnum() or c in ['_', '-']).strip()

In [ ]:
print("Scarico JSON dei team...")

# 1. Leggi il CSV delle squadre creato in precedenza
try:
    with open(input_csv_path, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        
        # Iteriamo su ogni squadra presente nel CSV
        for row in reader:
            id_squadra = row['ID Univoco']
            nome_squadra = row['Nome Squadra']
            
            print(f"Scarico ----------> {nome_squadra} (ID: {id_squadra})...")

            nome_squadra = clean_team_name(nome_squadra)
            file_path = f".\\Datasets\\PlayerStats\\TeamsJSONs\\{nome_squadra}_{id_squadra}_team.json"
            
            # URL dell'API ufficiale/pulita di FotMob per la squadra
            url = f"https://www.fotmob.com/_next/data/0Sm3sug45d5kO4fly3s6V/it/teams/{id_squadra}/overview/{nome_squadra}.json?lng=it&id={id_squadra}&tab=overview&slug={nome_squadra}"
            
            try:
                response = requests.get(url, headers=headers)
        
                if response.status_code == 200:
                    match_json = response.json()
                    
                    # Salvataggio del file JSON
                    with open(file_path, "w", encoding="utf-8") as out_file:
                        import json
                        json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                        
                elif response.status_code == 404:
                    print(f"Dettagli non trovati (404) per la squadra {nome_squadra}")
                else:
                    print(f"Errore {response.status_code} per la squadra {nome_squadra}")
            
            except Exception as e:
                print(f"Errore durante la richiesta per la squadra {nome_squadra}: {e}")
            
            # Pausa di 1 secondo tra le richieste per rispettare i server ed evitare ban ip
            time.sleep(1)

except FileNotFoundError:
    print(f"Errore: Il file '{input_csv_path}' non esiste. Esegui prima il primo script.")

Scarico JSON dei team...
Scarico ----------> Inter (ID: 8636)...
Scarico ----------> Napoli (ID: 9875)...
Scarico ----------> Roma (ID: 8686)...
Scarico ----------> Como (ID: 10171)...
Scarico ----------> Milan (ID: 8564)...
Scarico ----------> Juventus (ID: 9885)...
Scarico ----------> Atalanta (ID: 8524)...
Scarico ----------> Bologna (ID: 9857)...
Scarico ----------> Lazio (ID: 8543)...
Scarico ----------> Udinese (ID: 8600)...
Scarico ----------> Sassuolo (ID: 7943)...
Scarico ----------> Torino (ID: 9804)...
Scarico ----------> Parma (ID: 10167)...
Scarico ----------> Cagliari (ID: 8529)...
Scarico ----------> Fiorentina (ID: 8535)...
Scarico ----------> Genoa (ID: 10233)...
Scarico ----------> Lecce (ID: 9888)...
Scarico ----------> Cremonese (ID: 7801)...
Scarico ----------> Hellas Verona (ID: 9876)...
Scarico ----------> Pisa (ID: 6479)...


### Now I read all the JSON files and create an Excel file containing all the Serie A players

In [ ]:
cartella_json = r'.\Datasets\PlayerStats\TeamsJSONs'
output_excel_path = r'giocatori_serie_A_dettagliato.xlsx'

oggi = datetime.now()

In [18]:
def calcola_eta(dob_str):
    #formato passato YYYY-MM-DD

    if not dob_str:
        return 'N/D'
    try:
        data_nascita = datetime.strptime(dob_str.strip(), '%Y-%m-%d')
        
        eta = oggi.year - data_nascita.year - ((oggi.month, oggi.day) < (data_nascita.month, data_nascita.day))
        return eta
    except Exception:
        return 'N/D'

In [ ]:
# per debugging
ROSSO = "\033[91m"
RESET = "\033[0m"

# Lista globale per raccogliere i dati di tutti i giocatori
tutti_i_giocatori = []

# prendo tutti i json della cartella
file_json_squadre = [f for f in os.listdir(cartella_json) if f.endswith('_team.json')]

print(f"Trovati {len(file_json_squadre)} file JSON da elaborare.\n")

for file_name in file_json_squadre:
    # Estrae il nome della squadra e id dal nome del file (es. "Inter_8636_team.json" -> "Inter", "8636")
    team = file_name.replace('_team.json', '').replace('-', ' ')
    team_and_id = team.rsplit('_', 1)
    nome_team = team_and_id[0]
    id_team = team_and_id[1]
    file_path = os.path.join(cartella_json, file_name)
    
    print(f"Elaborazione: {file_name}...")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # Navigazione: pageProps -> fallback
        page_props = data.get('pageProps', {})
        fallback = page_props.get('fallback', {})
        
        chiave_team = f"team-{id_team}"
        
        # navigazione fino a squad -> squad
        team_data = fallback.get(chiave_team, {})
        squad_root = team_data.get('squad', {})
        squad_list = squad_root.get('squad') if isinstance(squad_root, dict) else squad_root
        
        # Valuto se esiste il team
        if not squad_list or not isinstance(squad_list, list):
            print(f"\033[91m -> [Salto File] Struttura squad non valida in {file_name}\033[0m")
            continue
            
        # Ciclo sui reparti (Portieri, Difensori, ecc.)
        for reparto in squad_list:
            if not isinstance(reparto, dict):
                continue

            if reparto.get('title') == 'coach':
                continue

            giocatori = reparto.get('members', [])
            ruolo_reparto = reparto.get('title', 'N/D')
            
            for p in giocatori:
                id_giocatore = p.get('id')
                name_giocatore = p.get('name')
                
                if not id_giocatore or not name_giocatore:
                    continue
                
                # Calcolo età
                dob = p.get('dateOfBirth', 'N/D')
                eta = calcola_eta(dob)

                role_obj = p.get('role')
                if isinstance(role_obj, dict):
                    role = role_obj.get('fallback', ruolo_reparto)
                else:
                    role = role_obj if role_obj else ruolo_reparto
                
                # Valore di mercato con la nuova formattazione numerica
                transfer_value = p.get('transferValue')
                
                # Estrazione Rating e Cartellini
                rating = p.get('rating')
                if rating is None:
                    rating = 'N/D'
                
                stats = p.get('stats', {})
                rcards = p.get('rcards', 0)
                ycards = p.get('ycards', 0)
                
                # Costruisco riga del giocatore
                tutti_i_giocatori.append({
                    'codice team': id_team,
                    'nome team': nome_team,
                    'id': id_giocatore,
                    'name': name_giocatore,
                    'dateOfBirth': dob if dob else 'N/D',
                    'età': eta,
                    'role': role,
                    'rating': rating,
                    'transferValue': transfer_value,
                    'rcards': rcards,
                    'ycards': ycards
                })
                
    except json.JSONDecodeError:
        print(f"{ROSSO} -> [Errore] File {file_name} corrotto o non è un JSON valido.{RESET}")
    except Exception as e:
        print(f"{ROSSO} -> [Errore] Problema imprevisto su {file_name}: {e}{RESET}")

Trovati 20 file JSON da elaborare.

Elaborazione: Atalanta_8524_team.json...
Elaborazione: Bologna_9857_team.json...
Elaborazione: Cagliari_8529_team.json...
Elaborazione: Como_10171_team.json...
Elaborazione: Cremonese_7801_team.json...
Elaborazione: Fiorentina_8535_team.json...
Elaborazione: Genoa_10233_team.json...
Elaborazione: Hellas-Verona_9876_team.json...
Elaborazione: Inter_8636_team.json...
Elaborazione: Juventus_9885_team.json...
Elaborazione: Lazio_8543_team.json...
Elaborazione: Lecce_9888_team.json...
Elaborazione: Milan_8564_team.json...
Elaborazione: Napoli_9875_team.json...
Elaborazione: Parma_10167_team.json...
Elaborazione: Pisa_6479_team.json...
Elaborazione: Roma_8686_team.json...
Elaborazione: Sassuolo_7943_team.json...
Elaborazione: Torino_9804_team.json...
Elaborazione: Udinese_8600_team.json...


In [ ]:
# creo file excel usando openpyxl
if tutti_i_giocatori:
    wb = Workbook()
    ws = wb.active
    ws.title = "Dati Giocatori Serie A"
    
    ws.views.sheetView[0].showGridLines = True
    
    # Intestazioni delle colonne
    headers = [
        'Codice Team', 'Nome Team', 'ID Giocatore', 'Nome Giocatore', 
        'Data di Nascita', 'Età', 'Ruolo', 'Rating', 'Valore di Mercato', 
        'Cartellini Rossi', 'Cartellini Gialli'
    ]
    ws.append(headers)
    
    for g in tutti_i_giocatori:
        # Estraiamo i valori dal dizionario nell'ordine esatto delle colonne dell'Excel
        riga_ordinata = [
            int(g['codice team']) if str(g['codice team']).isdigit() else g['codice team'],
            g['nome team'],
            g['id'],
            g['name'],
            g['dateOfBirth'],
            g['età'],
            g['role'],
            g['rating'],
            g['transferValue'],
            g['rcards'],
            g['ycards']
        ]
        ws.append(riga_ordinata)
        
    
    header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    header_fill = PatternFill(start_color='203764', end_color='203764', fill_type='solid')
    header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    

    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )
    
    ws.row_dimensions[1].height = 28
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = thin_border
        
    align_center = Alignment(horizontal='center', vertical='center')
    align_left = Alignment(horizontal='left', vertical='center')
    align_right = Alignment(horizontal='right', vertical='center')
    font_dati = Font(name='Segoe UI', size=10)
    
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=11):
        
        ws.row_dimensions[row[0].row].height = 20
        
        for cell in row:
            cell.font = font_dati
            cell.border = thin_border
            
            col_idx = cell.column
            
            if col_idx in [1, 3, 5, 6, 7]:
                cell.alignment = align_center
                if col_idx in [1, 3]:
                    cell.number_format = '@'
                    
            elif col_idx in [2, 4]:
                cell.alignment = align_left
                
            elif col_idx == 8:
                cell.alignment = align_center
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '0.00'
                else:
                    cell.value = 'N/D'
                    
            elif col_idx == 9:
                cell.alignment = align_right
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0" €"'
                else:
                    cell.value = 'N/D'
                    
            elif col_idx in [10, 11]:
                cell.alignment = align_center
                if isinstance(cell.value, int):
                    cell.number_format = '#,##0'

    ws.freeze_panes = 'A2'
    
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value:
                val_str = str(cell.value)
                if cell.column == 9 and isinstance(cell.value, (int, float)):
                    val_str = f"€ {val_str}00" 
                max_len = max(max_len, len(val_str))
        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)
        
        
    ws.parent.save(f".\\Datasets\\PlayerStats\\{output_excel_path}")
    print(f"\nEstrazione completata con successo!")
    print(f"I dati di {len(tutti_i_giocatori)} giocatori sono stati salvati in Excel: {output_excel_path}")
else:
    print(f"\033[91m\nNessun dato estratto. Controlla la posizione dei file JSON.\033[0m")


Estrazione completata con successo!
I dati di 623 giocatori sono stati salvati in Excel: giocatori_serie_A_dettagliato.xlsx


## 3. We can go into more detail now and download the JSON of each player through this link

https://www.fotmob.com/_next/data/0Sm3sug45d5kO4fly3s6V/it/players/CODICE/NOME-PARSATO.json?lng=it&id=CODICE&slug=NOME-PARSATO

<b>WARNING!!</b><br>
The link changes every so often:<br>
https://www.fotmob.com/_next/data/rUWhshM_9ZHzNFd42zY9J/it/players/CODICE/NOME-PARSATO.json?lng=it&id=CODICE&slug=NOME-PARSATO<br>
A buildID is used, which changes either periodically or for each session. To make the code work, I need to find a way to construct the link by finding the buildID first.

In [ ]:
input_excel_path = r'.\Datasets\PlayerStats\giocatori_serie_A_dettagliato.xlsx'

In [35]:
cookie_sporco = r"""g_state={"i_l":3,"i_ll":1780925656627,"i_b":"MFf4H6bXRxnKLQ5TXPFYcIhfteZROjqmKWt4yjEgqKk","i_e":{"enable_itp_optimization":0},"i_et":1779710661468,"i_p":1781364585107}; _hjSessionUser_2585474=eyJpZCI6ImNlMmRkOTllLWE5NDMtNWFjYS05YzE3LTI2MTQxNTdhMTRlOSIsImNyZWF0ZWQiOjE3Nzk3MTA2NjE2MzQsImV4aXN0aW5nIjp0cnVlfQ==; FCCDCF=%5Bnull%2Cnull%2Cnull%2C%5B%22CQkw6oAQkw6oAEsACBITCgFoAP_gAEPgABpwLItB_C7dbWFDyLZ3absEeIhHx_hjasQwAAbAAmAFTBqQoJQGwmEYNAyCtCACGAAAKmTBIQIkCAAAAUAAIIAFIADMAAyQIBAIICBAiAEJAEBACEACCoAgUQCIgIJEFEQAm…rope%2FRome%22%7D; panoramaId_expiry=1780991493427; turnstile_verified=1.1780909068.5f816827ee25910224c1c702fba7b7f473489bca54212549d589089a5641d5ed; _hjSession_2585474=eyJpZCI6IjcxNDlhOGRhLTBlM2MtNDRmNC05M2IyLTI3MjdkNDA3ZjUzYiIsImMiOjE3ODA5MjM1Njc4NjIsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowLCJzcCI6MH0=; FCNEC=%5B%5B%22AKsRol-LZiDKWjyz7sj32Pro57SKyT3lSV2UBUo-lSWzItrvpdxLB1_dEYGqwMmKKakL9U25j5CqB2XjFCZP3F_GqIoRUsda5Vqi4pGy6n5qHIr64IzNFgw8Le9I_S5ncGH2fxfZSzvl9zW1MPuWBhUjirfiViqG1Q%3D%3D%22%5D%5D"""
cookie_pulito = clean_header_string(cookie_sporco)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Alt-Used": "www.fotmob.com",
    "Cookie": cookie_pulito
}

In [ ]:
try:
    df = pd.read_excel(input_excel_path, engine='openpyxl')
    
    print(f"File caricato con successo! Trovati {len(df)} giocatori.\n")

except FileNotFoundError:
    print(f"\033[91m[Errore] Impossibile trovare il file '{input_excel_path}'. Verifica il percorso.\033[0m")
except Exception as e:
    print(f"\033[91m[Errore] Si è verificato un problema durante la lettura del file: {e}\033[0m")


File caricato con successo! Trovati 623 giocatori.



In [ ]:
def clear_player_name(text):
    if not text:
        return ""
    
    # tutto in minuscolo
    text_clean = str(text).lower().strip()
    
    # spazi con il trattino
    text_clean = text_clean.replace(" ", "-")
    
    # tolgo i caratteri accentati. Es: "à" diventa "a" + "accen_grave"
    text_normalized = unicodedata.normalize('NFD', text_clean)
    
    # tenieamo la stringa solo con i caratteri ASCII base
    slug = "".join(
        c for c in text_normalized 
        if (c.isalnum() and unicodedata.category(c) != 'Mn') or c in ['_', '-']
    )
    
    # tolgo trattini doppi
    while '--' in slug:
        slug = slug.replace('--', '-')
        
    return slug.strip('-')

In [38]:
#test
clear_player_name("João Mário")

'joao-mario'

### Search for the BuildID with Beautifulsoup

We have to run the following script every time we get an error 404 

In [ ]:
def extractNewBuildID():

    try:
        # faccio req alla home del sito
        response_home = requests.get("https://www.fotmob.com/it", headers=headers)
        soup = BeautifulSoup(response_home.text, 'html.parser')
        
        # Next.js nasconde i dati di configurazione nel tag script "__NEXT_DATA__"
        script_tag = soup.find('script', id='__NEXT_DATA__')
        
        if script_tag:
            next_data = json.loads(script_tag.string)
            build_id = next_data.get('buildId')
            print(f"\033[92mBuild ID trovato con successo: {build_id}\033[0m\n")
        else:
            raise ValueError("Tag __NEXT_DATA__ non trovato nella homepage.")
            
    except Exception as e:
        print(f"\033[91mImpossibile recuperare il Build ID. Errore: {e}\033[0m")
        exit()
    
    return build_id

extractNewBuildID()

Build ID trovato con successo: M7sn4_gVsnbrj9TotIYly



'M7sn4_gVsnbrj9TotIYly'

In [ ]:
print("Inizio download JSON giocatori:\n")

log = []

for index, riga in df.iterrows():
    
    id_giocatore = riga['ID Giocatore']
    nome_giocatore = riga['Nome Giocatore']
    
    clear_name = clear_player_name(nome_giocatore)
    
    file_name = f".\\Datasets\\PlayerStats\\PlayersJSONs\\{clear_name}_{id_giocatore}_player.json"
    
    max_tentativi = 3
    tentativo_attuale = 0
    scaricato = False      
    
    # URL dell'API ufficiale/pulita di FotMob per la squadra
    
    while tentativo_attuale < max_tentativi and not scaricato:

        url = f"https://www.fotmob.com/_next/data/{build_id}/it/players/{id_giocatore}/{clear_name}.json?lng=it&id={id_giocatore}&slug={clear_name}"
    
        try:
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                match_json = response.json()
                
                # Salvataggio del file JSON locale
                with open(file_name, "w", encoding="utf-8") as out_file:
                    import json
                    json.dump(match_json, out_file, ensure_ascii=False, indent=4)
                    
                print(f"[{index+1}/{len(df)}] Scaricato: {nome_giocatore}")
                scaricato = True

            elif response.status_code == 404:
                # Se risposta è 404 allora faccio ripartire lo script
                tentativo_attuale += 1
                print(f"\033[91m[{index+1}/{len(df)}] Dettagli non trovati (404) per: {nome_giocatore}. Verifica se lo slug è esatto.\033[0m")
                build_id = extractNewBuildID()
                time.sleep(1)
            else:
                print(f"\033[91m[{index+1}/{len(df)}] Errore {response.status_code} per: {nome_giocatore}\033[0m")
        
        except Exception as e:
            print(f"\033[91mErrore di connessione per {nome_giocatore}: {e}\033[0m")

    if not scaricato:
        string_log = f"Impossibile scaricare {nome_giocatore} dopo {max_tentativi} tentativi."
        log.append(string_log)
        print(f"\033[91m-> Impossibile scaricare {nome_giocatore} dopo {max_tentativi} tentativi. Passo al prossimo.\033[0m")
        
    # Pausa di 1 secondo tra le richieste per rispettare i server ed evitare ban ip
    time.sleep(1)


Inizio download JSON giocatori:

[1/623] Scaricato: Marco Carnesecchi
[2/623] Scaricato: Francesco Rossi
[3/623] Scaricato: Marco Sportiello
[4/623] Scaricato: Odilon Kossounou
[5/623] Scaricato: Isak Hien
[6/623] Scaricato: Berat Djimsiti
[7/623] Scaricato: Sead Kolasinac
[8/623] Scaricato: Giorgio Scalvini
[9/623] Scaricato: Honest Ahanor
[10/623] Scaricato: Mitchel Bakker
[11/623] Scaricato: Yunus Musah
[12/623] Scaricato: Kamaldeen Sulemana
[13/623] Scaricato: Mario Pasalic
[14/623] Scaricato: Lazar Samardzic
[15/623] Scaricato: Éderson
[16/623] Scaricato: Marten de Roon
[17/623] Scaricato: Raoul Bellanova
[18/623] Scaricato: Charles De Ketelaere
[19/623] Scaricato: Giacomo Raspadori
[20/623] Scaricato: Lorenzo Bernasconi
[21/623] Scaricato: Nicola Zalewski
[22/623] Scaricato: Davide Zappacosta
[23/623] Scaricato: Gianluca Scamacca
[24/623] Scaricato: Giulio Misitano
[25/623] Scaricato: Nikola Krstovic
[26/623] Scaricato: Lukasz Skorupski
[27/623] Scaricato: Federico Ravaglia
[28/6

In [52]:
for entry in log:
    print(f"\033[91m{entry}\033[0m")

### Let's read all the JSON files and compose the final player dataset

I will use Pandas to merge the data I find in the JSON files with the data I already have in the Excel file.

In [ ]:
percorso_excel = r'.\Datasets\PlayerStats\giocatori_serie_A_dettagliato.xlsx'
folder_json = r'.\Datasets\PlayerStats\PlayersJSONs'

df_excel = pd.read_excel(percorso_excel)

# trasformo la stringa "N/D" o altri valori vuoti in NaN (serve a Pandas)
# lo 0 è perche molti cartellini non hanno un valore e sono segnati come 0
# Es. tutti i giocatori dell'atalanta hanno 0 cartellini (gialli e rossi), cosa impossibile
df_excel = df_excel.replace(['N/D', 'null', '', 0], pd.NA)

# ID Giocatore come indice del DataFrame
df_excel.set_index('ID Giocatore', inplace=True)

log = []
dati_json = []

# prendo u file alla volta dalla cartella
for filename in os.listdir(folder_json):
    if filename.endswith('.json'):

        # costruisco il filepath per aprire il json
        filepath = os.path.join(folder_json, filename)
        
        with open(filepath, 'r', encoding='utf-8') as file:
            try:
                json_content = json.load(file)
                player_data = json_content.get('pageProps', {}).get('data', {})
                
                if not player_data:
                    log.append(f"Struttura JSON non valida (NO PLAYER DATA) in --> {filename}")
                    continue
                    
                player_id = player_data.get('id')
                
                # Inizializzo un dizionario utilizzando l'ID
                player_dict = {
                    'ID Giocatore': player_id
                }
                
                # Estrai le statistiche
                stats = player_data.get('mainLeague', {}).get('stats')
                if stats:
                    for stat in stats:
                        title = stat.get('title')
                        value = stat.get('value')
                        if title:
                            player_dict[title] = value
                else:
                    log.append(f"Nessuna statistica trovata per --> {filename}")
                            
                dati_json.append(player_dict)
                
            except Exception as e:
                print(f"Errore durante la lettura del file: {e} --> {filename}")

### Verifico per quali giocatori non ci sono statistiche o il file non ha il formato corretto

In [23]:
for i in log:
    print(i)

Nessuna statistica trovata per --> adi-kurti_1805110_player.json
Nessuna statistica trovata per --> adrian-lickunas_1841063_player.json
Nessuna statistica trovata per --> adrian-malovec_1553484_player.json
Nessuna statistica trovata per --> alen-sherri_1178779_player.json
Nessuna statistica trovata per --> alessandro-ciardi_1663329_player.json
Nessuna statistica trovata per --> alessandro-mazzocchi_1997571_player.json
Nessuna statistica trovata per --> alessandro-nunziante_1477567_player.json
Nessuna statistica trovata per --> alessio-chiesa_1695978_player.json
Nessuna statistica trovata per --> alessio-raballo_1702418_player.json
Nessuna statistica trovata per --> andrej-popovic_1901440_player.json
Nessuna statistica trovata per --> antoni-franke_1817271_player.json
Nessuna statistica trovata per --> benjamin-siegrist_265006_player.json
Nessuna statistica trovata per --> bernardo-conde_1853664_player.json
Nessuna statistica trovata per --> bodin-tomasevic_1659853_player.json
Nessuna s

In [24]:
print(len(log))

99


dataframe columns:<br>
ID Giocatore | Goals | Assists | Started | Matches | Minutes played | Rating | Yellow cards | Red cards | Clean sheets | Goals conceded | Saved penalties

I change the file excel intestations or the merge will not woek

In [ ]:

#cro DataFrame con i dati dei JSON
df_json = pd.DataFrame(dati_json)

#utilizzo combine_first (merge intelligente)
if not df_json.empty:
    df_json.set_index('ID Giocatore', inplace=True)
    df_finale = df_excel.combine_first(df_json)
else:
    print("Nessun dato valido trovato nei JSON.")
    df_finale = df_excel

# faccio tornare 'ID Giocatore' una colonna normale
df_finale.reset_index(inplace=True)

# con questo sostituisco i Nan, ma sarebbe meglio tenerli così per il training
df_finale = df_finale.fillna('N/D')

df_finale.to_excel(r'.\Datasets\STATS_giocatori_serie_A.xlsx', index=False)

print("Dati uniti con successo! Ecco un'anteprima:")
display(df_finale)

Dati uniti con successo! Ecco un'anteprima:


,ID Giocatore,Codice Team,Nome Team,Nome Giocatore,Data di Nascita,Età,Ruolo,Rating,Valore di Mercato,Red Cards,...,Goals,Assists,Started,Matches,Minutes played,Yellow cards,Red cards,Clean sheets,Goals conceded,Saved penalties
0,25587,9857,Bologna,Lorenzo De Silvestri,1988-05-23,38,Defender,6.49,298808,N/D,...,0.0,0.0,6.0,12.0,491.0,2.0,0.0,N/D,N/D,N/D
1,31097,8564,Milan,Luka Modric,1985-09-09,40,Midfielder,7.58,2177851,N/D,...,2.0,3.0,32.0,34.0,2816.0,4.0,0.0,N/D,N/D,N/D
2,34029,6479,Pisa,Raúl Albiol,1985-09-04,40,Defender,6.61,512173,N/D,...,0.0,1.0,5.0,8.0,502.0,1.0,0.0,N/D,N/D,N/D
3,41618,8636,Inter,Yann Sommer,1988-12-17,37,Keeper,6.84,1785639,N/D,...,N/D,N/D,N/D,33.0,2961.0,0.0,0.0,15.0,30.0,0/4
4,42692,8600,Udinese,Daniele Padelli,1985-10-25,40,Keeper,6.88,343895,N/D,...,N/D,N/D,N/D,2.0,180.0,0.0,0.0,N/D,2.0,0/1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
618,1946408,7801,Cremonese,Nouroudine Faye,2007-01-03,19,Attacker,N/D,15000,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D
619,1948559,10233,Genoa,Latif Ouedraogo,2007-10-23,18,Defender,6.34,85000,N/D,...,0.0,0.0,0.0,2.0,49.0,0.0,0.0,N/D,N/D,N/D
620,1997571,10167,Parma,Alessandro Mazzocchi,2008-04-08,18,Keeper,N/D,10000,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D
621,1997622,10233,Genoa,Elia Spicuglia,2007-06-19,18,Attacker,N/D,35000,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D,N/D


### Style for readibility

In [ ]:
#creazione workbook con openpyxl
wb = openpyxl.Workbook()

ws_data = wb.active
ws_data.title = "Risultati e Statistiche"
ws_data.views.sheetView[0].showGridLines = True


# Stile pagina
navy_fill = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
light_blue_fill = PatternFill(start_color="F2F5F9", end_color="F2F5F9", fill_type="solid")
white_font = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
regular_font = Font(name="Segoe UI", size=10)
bold_font = Font(name="Segoe UI", size=10, bold=True)
title_font = Font(name="Segoe UI", size=16, bold=True, color="1B365D")
section_font = Font(name="Segoe UI", size=12, bold=True, color="1B365D")

thin_gray = Side(style='thin', color='D3D3D3')
cell_border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)


# Populate Data Sheet
headers = list(df_finale.columns)
ws_data.append(headers)
for row in ws_data[1]:
    row.fill = navy_fill
    row.font = white_font
    row.alignment = Alignment(horizontal="center", vertical="center")

for idx, r_data in df_finale.iterrows():
    row_values = [r_data[h] for h in headers]
    ws_data.append(row_values)
    row_num = idx + 2
    
    
    # Zebra striping
    if row_num % 2 == 1:
        for col_idx in range(1, len(headers) + 1):
            c = ws_data.cell(row=row_num, column=col_idx)
            c.fill = light_blue_fill

    # Borders and alignment
    for col_idx in range(1, len(headers) + 1):
        c = ws_data.cell(row=row_num, column=col_idx)
        c.border = cell_border
        if col_idx in [1, 2, 3, 6, 7, 8, 9]:
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.alignment = Alignment(horizontal="left", vertical="center")

# Auto-adjust column widths
for col in ws_data.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)


ws_data.freeze_panes = "A2"
path_stats_styled = r".\Datasets\Styled_STATS_giocatori_serie_A.xlsx"
wb.save(path_stats_styled)

# Check

In [30]:
nd_per_feature = (df_finale == 'N/D').sum()
print(nd_per_feature)

ID Giocatore           0
Codice Team            0
Nome Team              0
Nome Giocatore         0
Data di Nascita        0
Età                    0
Ruolo                  0
Rating               113
Valore di Mercato     12
Red Cards            608
Yellow Cards         558
Goals                146
Assists              146
Started              146
Matches               99
Minutes played        99
Yellow cards          99
Red cards             99
Clean sheets         587
Goals conceded       581
Saved penalties      594
dtype: int64
